# Lecture 4: Regular Expressions & Structured Text
### NLP Course 2027

---

## Learning Outcomes
- Write regex patterns for text extraction and cleaning
- Use Python's `re` module for NLP preprocessing
- Build a custom tokenizer with regex
- Apply chunking patterns for syntactic analysis

**Primary Reference:** *NLP with Python* Ch.3 (regex section), Ch.7 (chunking)

## 1. Why Regex in NLP?

Regular expressions are **patterns** that describe sets of strings.
In NLP, regex is used for:
- Text **cleaning** (remove HTML, URLs, extra spaces)
- **Extraction** (emails, phone numbers, dates, prices)
- **Tokenization** (custom splitting rules)
- **Chunking** (simple syntactic patterns)
- **Validation** (checking format of extracted entities)

```
Text: 'Call us at +1-800-555-0100 or email info@example.com'
Regex: r'[\w.+-]+@[\w-]+\.[\w.-]+'
Match: 'info@example.com'
```

In [8]:
import re

# Basic regex functions
text = 'The price is $42.99 and $100.50 for the NLP course in 2024.'

# re.search: find first match
m = re.search(r'\d+\.\d+', text)
print('First number:', m.group() if m else 'not found')

# re.findall: all matches
prices = re.findall(r'\$\d+\.\d+', text)
print('All prices:', prices)

# re.sub: replace matches
clean = re.sub(r'\$\d+\.\d+', '[PRICE]', text)
print('Cleaned:', clean)

First number: 42.99
All prices: ['$42.99', '$100.50']
Cleaned: The price is [PRICE] and [PRICE] for the NLP course in 2024.


## 2. Regex Syntax Reference

### Character Classes
| Pattern | Matches |
|---------|--------|
| `\d` | Any digit [0-9] |
| `\w` | Word character [a-zA-Z0-9_] |
| `\s` | Whitespace [space, tab, newline] |
| `.` | Any character (except newline) |
| `[aeiou]` | Any vowel |
| `[^aeiou]` | Any non-vowel |

### Quantifiers
| Pattern | Meaning |
|---------|--------|
| `*` | 0 or more |
| `+` | 1 or more |
| `?` | 0 or 1 (optional) |
| `{n}` | Exactly n |
| `{n,m}` | Between n and m |

### Anchors & Groups
| Pattern | Meaning |
|---------|--------|
| `^` | Start of string/line |
| `$` | End of string/line |
| `\b` | Word boundary |
| `(...)` | Capturing group |
| `(?:...)` | Non-capturing group |
| `a\|b` | Either a or b |

In [9]:
# Common NLP extraction patterns
import re

test_text = """
Contact: john.doe@email.com or jane_smith@company.org
Phone: +1-800-555-0100 or (415) 555-2671
Website: https://www.example.com or http://nlp.org/course
Date: 2024-01-15 or January 15, 2024 or 01/15/2024
Price: $42.99, €100, £29.50
Hashtags: #NLP #MachineLearning #AI
"""

patterns = {
    'email':   r'[\w.+-]+@[\w-]+\.[\w.-]+',
    'url':     r'https?://[^\s]+',
    'phone':   r'(?:\+1[-.])?\(?\d{3}\)?[-.]\d{3}[-.]\d{4}',
    'date':    r'\d{4}-\d{2}-\d{2}',
    'price':   r'[\$€£]\d+(?:\.\d+)?',
    'hashtag': r'#\w+',
}

for name, pattern in patterns.items():
    matches = re.findall(pattern, test_text)
    print(f'{name:10s}: {matches}')

email     : ['john.doe@email.com', 'jane_smith@company.org']
url       : ['https://www.example.com', 'http://nlp.org/course']
phone     : ['+1-800-555-0100']
date      : ['2024-01-15']
price     : ['$42.99', '€100', '£29.50']
hashtag   : ['#NLP', '#MachineLearning', '#AI']


In [10]:
# Text cleaning with regex

def clean_text(text):
    """Comprehensive text cleaning pipeline."""
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove email addresses
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '', text)
    # Remove special characters (keep letters, digits, basic punct)
    text = re.sub(r'[^\w\s.,!?]', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

noisy = """<b>Check out</b> our website at https://example.com!
Contact admin@test.org for info. Price: $99 & $49!!!
We're #1 in NLP   &   Machine Learning."""

print('Original:', noisy)
print('Cleaned: ', clean_text(noisy))

Original: <b>Check out</b> our website at https://example.com!
Contact admin@test.org for info. Price: $99 & $49!!!
We're #1 in NLP   &   Machine Learning.
Cleaned:  Check out our website at Contact for info. Price 99 49!!! Were 1 in NLP Machine Learning.


In [11]:
# Named groups for structured extraction

# Extract structured date components
date_pattern = r'(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})'
date_str = 'Published: 2024-03-15'

m = re.search(date_pattern, date_str)
if m:
    print('Year:', m.group('year'))
    print('Month:', m.group('month'))
    print('Day:', m.group('day'))
    print('Full match:', m.group())

# Extract all named entities from a structured format
text = 'Name: John Smith; Age: 30; City: New York; Job: Data Scientist'
pairs = re.findall(r'(\w+(?:\s\w+)?): ([\w\s]+?)(?:;|$)', text)
print('\nExtracted pairs:', pairs)

Year: 2024
Month: 03
Day: 15
Full match: 2024-03-15

Extracted pairs: [('Name', 'John Smith'), ('Age', '30'), ('City', 'New York'), ('Job', 'Data Scientist')]


In [12]:
# Custom Tokenizer with RegexpTokenizer
from nltk.tokenize import RegexpTokenizer

# Default: words only
word_tok = RegexpTokenizer(r'\w+')

# Preserves apostrophes (contractions)
contract_tok = RegexpTokenizer(r"\w+(?:'\w+)*")

# Handles emoticons and hashtags
tweet_pattern = r"#\w+|@\w+|:\)|:-\)|:\(|https?://\S+|\w+(?:'\w+)*"
tweet_tok = RegexpTokenizer(tweet_pattern)

texts = [
    "Hello! I can't believe it's so easy.",
    "The #NLP course is amazing! @professor :) Check: https://nlp.org"
]

for text in texts:
    print(f'Text: {text}')
    print(f'  word_tok:     {word_tok.tokenize(text)}')
    print(f'  contract_tok: {contract_tok.tokenize(text)}')
    print(f'  tweet_tok:    {tweet_tok.tokenize(text)}')
    print()

Text: Hello! I can't believe it's so easy.
  word_tok:     ['Hello', 'I', 'can', 't', 'believe', 'it', 's', 'so', 'easy']
  contract_tok: ['Hello', 'I', "can't", 'believe', "it's", 'so', 'easy']
  tweet_tok:    ['Hello', 'I', "can't", 'believe', "it's", 'so', 'easy']

Text: The #NLP course is amazing! @professor :) Check: https://nlp.org
  word_tok:     ['The', 'NLP', 'course', 'is', 'amazing', 'professor', 'Check', 'https', 'nlp', 'org']
  contract_tok: ['The', 'NLP', 'course', 'is', 'amazing', 'professor', 'Check', 'https', 'nlp', 'org']
  tweet_tok:    ['The', '#NLP', 'course', 'is', 'amazing', '@professor', ':)', 'Check', 'https://nlp.org']



## 3. Regex for Chunking (Shallow Parsing)

**Chunking** identifies non-overlapping phrases using POS tag patterns.

```
Sentence:   The  quick  brown  fox  jumped
POS tags:   DT   JJ     JJ     NN   VBD
Chunk:      └──── NP ────┘
```

Grammar format:
```
NP: {<DT>?<JJ>*<NN.*>}   # optional DT, any JJs, then a noun
VP: {<VB.*><NP|PP>*}     # verb followed by NPs/PPs
```

In [13]:
import nltk
from nltk import RegexpParser, pos_tag, word_tokenize

sentence = 'The quick brown fox jumped over the lazy dog'
tagged = pos_tag(word_tokenize(sentence))
print('POS tagged:', tagged)
print()

# Define NP grammar
grammar = r"""
  NP: {<DT>?<JJ.*>*<NN.*>+}   # Noun Phrase
  PP: {<IN><NP>}                # Prepositional Phrase
"""
parser = RegexpParser(grammar)
tree = parser.parse(tagged)
print('Parse tree:')
print(tree)

POS tagged: [('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('fox', 'NN'), ('jumped', 'VBD'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN')]

Parse tree:
(S
  (NP The/DT quick/JJ brown/NN fox/NN)
  jumped/VBD
  (PP over/IN (NP the/DT lazy/JJ dog/NN)))


In [14]:
# Extract chunks as text spans
def extract_chunks(sentence, chunk_type='NP'):
    """Extract all chunks of a given type from a sentence."""
    grammar = r"""
      NP: {<DT>?<JJ.*>*<NN.*>+}
      VP: {<VB.*><RB>*}
    """
    tagged = pos_tag(word_tokenize(sentence))
    tree = RegexpParser(grammar).parse(tagged)
    
    chunks = []
    for subtree in tree.subtrees():
        if subtree.label() == chunk_type:
            chunk = ' '.join(w for w, t in subtree.leaves())
            chunks.append(chunk)
    return chunks

sentences = [
    'The large red car quickly accelerated down the steep hill.',
    'Natural Language Processing is a fascinating field of computer science.',
    'The researchers published three interesting papers about machine learning.'
]

for s in sentences:
    nps = extract_chunks(s, 'NP')
    print(f'Sentence: {s}')
    print(f'  NPs: {nps}')
    print()

Sentence: The large red car quickly accelerated down the steep hill.
  NPs: ['The large red car', 'the steep hill']

Sentence: Natural Language Processing is a fascinating field of computer science.
  NPs: ['Natural Language Processing', 'a fascinating field', 'computer science']

Sentence: The researchers published three interesting papers about machine learning.
  NPs: ['The researchers', 'interesting papers', 'machine learning']



## Practice Exercises

See **`Lecture-04-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Technique | Module/Function | Use Case |
|-----------|-----------------|----------|
| Pattern matching | `re.search`, `re.findall` | Find entities |
| Replacement | `re.sub` | Text cleaning |
| Custom tokenization | `RegexpTokenizer` | Domain-specific splitting |
| Chunking | `RegexpParser` | Shallow parsing |
| Named groups | `(?P<name>...)` | Structured extraction |

**Next Lecture**: Classification Fundamentals — Naive Bayes and feature extraction.

---
*Book references: NLP with Python Ch.3, 7 | Practical NLP Ch.2*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**